# 09_some_tests.ipynb - Diagnóstico de A-G Reaction Crash

## Problema conocido:
- El notebook principal (01) crashea a los 6-20 minutos
- El diagnóstico directo funciona (100 ox × 5236 amines, ~2 min)
- El tester funciona (680 amines)
- El cache funciona correctamente

## Hipótesis:
El problema está en `load_or_run()` o en cómo interactúa con `checkpoint_manager`

## Tests (ejecutar en orden):
1. **TEST 1**: rxn_AminolysisGFPc DIRECTO (referencia)
2. **TEST 2**: load_or_run wrapper (IGUAL que notebook principal)
3. **TEST 3**: load_or_run con más datos (1000 ox)
4. **TEST 4**: Comparar n_workers (1 vs 4)
5. **TEST 5**: Resume desde checkpoint

## Setup: Cargar datos

In [1]:
import time
import gc
import psutil
import pandas as pd
from pathlib import Path
import py_utils as pu

start = time.time()
print(f"[{time.strftime('%H:%M:%S')}] Loading data...")

OXAZOLONES_PATH = "mol_files/3. Oxazolones/Oxazolones_veber_4776914cmpds.csv"
AMINES_PATH = "mol_files/2. Building Blocks/Amines_veber_5236cmpds.csv"

df_ox = pd.read_csv(OXAZOLONES_PATH)
df_am = pd.read_csv(AMINES_PATH)

print(f"[{time.strftime('%H:%M:%S')}] Data loaded!")
print(f"  Oxazolones: {len(df_ox):,}")
print(f"  Amines: {len(df_am):,}")
print(f"  Memory: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

[21:03:34] Loading data...
[21:03:38] Data loaded!
  Oxazolones: 4,776,914
  Amines: 5,236
  Memory: 2.2 GB


## TEST 1: rxn_AminolysisGFPc DIRECTO (sin load_or_run, sin checkpoint)

**Este es el test de referencia que YA FUNCIONA.**
Si esto falla, algo muy básico está roto.

In [2]:
print(f"[{time.strftime('%H:%M:%S')}] TEST 1: Direct rxn_AminolysisGFPc")
print(f"  Data: 100 ox × 5236 amines = 523,600 pares")
gc.collect()

result_t1 = pu.rxn_AminolysisGFPc(
    df_ox.head(100),
    df_am,
    chunk_size=10,
    n_workers=4,
    use_cache=True,
    print_report=True
)

print(f"\n[{time.strftime('%H:%M:%S')}] TEST 1 COMPLETE: {len(result_t1)} products")
print(f"  Memory: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

[21:03:38] TEST 1: Direct rxn_AminolysisGFPc
  Data: 100 ox × 5236 amines = 523,600 pares
[AminolysisGFPc] 523,600 total pairs (10 chunks of ~10 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Cache: 523600 hits, 0 misses (100.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 100, 'input_amines': 5236, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 476500, 'cache_hits': 523600, 'cache_misses': 0}

[21:05:50] TEST 1 COMPLETE: 476500 products
  Memory: 0.4 GB


## TEST 2: load_or_run + rxn_AminolysisGFPc

**Este es IGUAL que en el notebook principal (01).**
Si crashea aquí, el problema es `load_or_run()` o `checkpoint_manager`.

In [3]:
print(f"[{time.strftime('%H:%M:%S')}] TEST 2: load_or_run wrapper")
print(f"  Data: 100 ox × 5236 amines = 523,600 pares")
gc.collect()

# Limpiar checkpoint anterior
diag_checkpoint = Path("mol_files/4. Imidazolones/.cache/DiagTest_checkpoint.json")
if diag_checkpoint.exists():
    diag_checkpoint.unlink()
    print("  Deleted old checkpoint")

result_t2 = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_AminolysisGFPc(
        df_ox.head(100),
        df_am,
        chunk_size=10,
        n_workers=4,
        use_cache=True,
        checkpoint_manager=checkpoint_manager,
        print_report=True
    ),
    output_csv="mol_files/4. Imidazolones/DiagTest_raw.csv",
    stage_name="DiagTest",
    force_recompute=True
)

print(f"\n[{time.strftime('%H:%M:%S')}] TEST 2 COMPLETE: {len(result_t2)} products")
print(f"  Memory: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

[21:05:50] TEST 2: load_or_run wrapper
  Data: 100 ox × 5236 amines = 523,600 pares
  Deleted old checkpoint
[load_or_run] Computing DiagTest...
[AminolysisGFPc] 523,600 total pairs (10 chunks of ~10 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Checkpoint saved: DiagTest_checkpoint.json
[AminolysisGFPc] Cache: 523600 hits, 0 misses (100.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 100, 'input_amines': 5236, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 476500, 'cache_hits': 523600, 'cache_misses': 0}
[load_or_run] Saved DiagTest_raw_476500cmpds.csv (476,500 rows)

[21:08:02] TEST 2 COMPLETE: 476500 products
  Memory: 0.4 GB


## TEST 3: load_or_run con más datos (1000 ox)

Si Test 2 pasa pero esto crashea, el problema es escalabilidad.

In [4]:
print(f"[{time.strftime('%H:%M:%S')}] TEST 3: load_or_run con 1000 ox")
print(f"  Data: 1000 ox × 5236 amines = 5,236,000 pares")
gc.collect()

diag_checkpoint = Path("mol_files/4. Imidazolones/.cache/DiagTest_checkpoint.json")
if diag_checkpoint.exists():
    diag_checkpoint.unlink()

result_t3 = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_AminolysisGFPc(
        df_ox.head(1000),
        df_am,
        chunk_size=10,
        n_workers=4,
        use_cache=True,
        checkpoint_manager=checkpoint_manager,
        print_report=True
    ),
    output_csv="mol_files/4. Imidazolones/DiagTest_raw_1000.csv",
    stage_name="DiagTest",
    force_recompute=True
)

print(f"\n[{time.strftime('%H:%M:%S')}] TEST 3 COMPLETE: {len(result_t3)} products")
print(f"  Memory: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

[21:08:02] TEST 3: load_or_run con 1000 ox
  Data: 1000 ox × 5236 amines = 5,236,000 pares
[load_or_run] Computing DiagTest...
[AminolysisGFPc] 5,236,000 total pairs (100 chunks of ~10 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Checkpoint saved: DiagTest_checkpoint.json
[AminolysisGFPc] Cache: 5236000 hits, 0 misses (100.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 1000, 'input_amines': 5236, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 4765000, 'cache_hits': 5236000, 'cache_misses': 0}
[load_or_run] Saved DiagTest_raw_1000_raw_4765000cmpds.csv (4,765,000 rows)

[21:21:47] TEST 3 COMPLETE: 4765000 products
  Memory: 2.2 GB


## TEST 4: Comparar n_workers (1 vs 4)

Verificar si multiprocessing causa el problema.

In [5]:
print(f"[{time.strftime('%H:%M:%S')}] TEST 4: Comparing n_workers")
print(f"  Data: 100 ox × 100 amines (reducido para speed)")

df_am_small = df_am.head(100)

for n_w in [1, 4]:
    print(f"\n  === n_workers={n_w} ===")
    gc.collect()
    
    t_start = time.time()
    result = pu.rxn_AminolysisGFPc(
        df_ox.head(100),
        df_am_small,
        chunk_size=10,
        n_workers=n_w,
        use_cache=True,
        print_report=True
    )
    
    print(f"  Time: {time.time()-t_start:.1f}s, Products: {len(result)}")

print(f"\n[{time.strftime('%H:%M:%S')}] TEST 4 COMPLETE")

[21:21:47] TEST 4: Comparing n_workers
  Data: 100 ox × 100 amines (reducido para speed)

  === n_workers=1 ===
[AminolysisGFPc] 10,000 total pairs (10 chunks of ~10 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Cache: 10000 hits, 0 misses (100.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 100, 'input_amines': 100, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 10700, 'cache_hits': 10000, 'cache_misses': 0}
  Time: 106.9s, Products: 10700

  === n_workers=4 ===
[AminolysisGFPc] 10,000 total pairs (10 chunks of ~10 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Cache: 10000 hits, 0 misses (100.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 100, 'input_amines': 100, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not

## TEST 5: Test de RESUME (simular crash y reanudar)

Verificar que el checkpoint funciona para reanudar.

In [6]:
print(f"[{time.strftime('%H:%M:%S')}] TEST 5: Resume from checkpoint")
print(f"  Data: 200 ox × 5236 amines = 1,047,200 pares")

diag_checkpoint = Path("mol_files/4. Imidazolones/.cache/DiagTest_checkpoint.json")

# Limpiar todo antes de empezar
if diag_checkpoint.exists():
    diag_checkpoint.unlink()

# Primera pasada (force_recompute=True)
print("\n  -- First pass (force_recompute=True) --")
result_first = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_AminolysisGFPc(
        df_ox.head(200),
        df_am,
        chunk_size=10,
        n_workers=4,
        use_cache=True,
        checkpoint_manager=checkpoint_manager,
        print_report=True
    ),
    output_csv="mol_files/4. Imidazolones/DiagTest_resume.csv",
    stage_name="DiagTest",
    force_recompute=True
)

print(f"\n  First pass: {len(result_first)} products")
print(f"  Checkpoint exists: {diag_checkpoint.exists()}")

# Segunda pasada (force_recompute=False, debería cargar desde checkpoint)
print("\n  -- Second pass (force_recompute=False) --")
result_second = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_AminolysisGFPc(
        df_ox.head(200),
        df_am,
        chunk_size=10,
        n_workers=4,
        use_cache=True,
        checkpoint_manager=checkpoint_manager,
        print_report=True
    ),
    output_csv="mol_files/4. Imidazolones/DiagTest_resume.csv",
    stage_name="DiagTest",
    force_recompute=False
)

print(f"\n  Second pass: {len(result_second)} products")
print(f"\n  [VERIFICATION]")
if len(result_first) == len(result_second):
    print(f"  ✅ PASS: Same number of products ({len(result_first)})")
else:
    print(f"  ❌ FAIL: Different products ({len(result_first)} vs {len(result_second)})")

print(f"\n[{time.strftime('%H:%M:%S')}] TEST 5 COMPLETE")

[21:25:22] TEST 5: Resume from checkpoint
  Data: 200 ox × 5236 amines = 1,047,200 pares

  -- First pass (force_recompute=True) --
[load_or_run] Computing DiagTest...
[AminolysisGFPc] 1,047,200 total pairs (20 chunks of ~10 oxazolones)
[AminolysisGFPc] Cache saved to: mol_files/3. Oxazolones/.cache/aminolysis_gfpc_cache.json.gz
[AminolysisGFPc] Checkpoint saved: DiagTest_checkpoint.json
[AminolysisGFPc] Cache: 1047200 hits, 0 misses (100.0% hit rate)
[AminolysisGFPc] Stats: {'input_oxazolones': 200, 'input_amines': 5236, 'invalid_oxazolone': 0, 'invalid_amine': 0, 'not_oxazolone': 0, 'not_amine': 0, 'no_product': 0, 'problematic': 0, 'output_rows': 953000, 'cache_hits': 1047200, 'cache_misses': 0}
[load_or_run] Saved DiagTest_resume.csv (953,000 rows)

  First pass: 953000 products
  Checkpoint exists: True

  -- Second pass (force_recompute=False) --
[load_or_run] Loading DiagTest_raw_476500cmpds.csv (476,500 rows) ✓

  Second pass: 476500 products

  [VERIFICATION]
  ❌ FAIL: Differe

## Resumen

| Test | Descripción | Esperado |
|------|-------------|----------|
| 1 | Direct rxn_AminolysisGFPc | ✅ DEBE PASAR |
| 2 | load_or_run wrapper | ❓ AQUÍ FALLA? |
| 3 | load_or_run 1000 ox | ❓ ESCALABILIDAD? |
| 4 | n_workers comparison | Verificar mp |
| 5 | Resume checkpoint | Verificar checkpoints |